> **⚠ External Data Required**
>
> This notebook requires two Excel files that are **not included** in this repository:
> - `demogr-filtered.xlsx` — demographic survey records with columns `ind`, `sex`, `age_sim`
> - `big5_data.xlsx` — Big Five personality survey records with columns `id`, `gender`, `age`
>
> To run this notebook, place both files in the same directory as this notebook (`notebooks/07_fuzzy_matching/`) and re-run all cells.

# Chapter 7.2 — Fuzzy Dataset Linking with `fuzzymatcher`

This notebook demonstrates linking two real-world datasets using the `fuzzymatcher` library, which uses a SQLite-backed TF-IDF index for efficient candidate retrieval, followed by a token-level similarity score to rank matches.

The two datasets share the key fields `sex`/`gender` and `age_sim`/`age`, but the field names and value codings differ — a common situation when integrating data from independent surveys.

**Two outputs are produced:**
- `link_table()` — a scored link table showing all candidate matches with their similarity scores.
- `fuzzy_left_join()` — a left join that appends the best-matching right-table record to each left-table record.

**When to use this approach:**
- The datasets to be linked are too large for the all-pairs Levenshtein approach in Chapter 7.1.
- You need a match score alongside the joined data for downstream filtering or auditing.

## Load Data

Replace the paths below with the locations of your Excel files.

In [37]:
import pandas as pd
from pathlib import Path
#import fuzzymatcher as fm
import fuzzynoisymatcher as fm
import sqlite3
from sqlite_fts4 import register_functions
import numpy as np
import pandas_profiling

## Preview Both Datasets

In [38]:
demogr = pd.read_excel('demogr-filtered.xlsx')
big5 = pd.read_excel('big5_data.xlsx')

In [39]:
demogr[0:5]

,ind,sex,age_sim,lv_educ,empl_stat,marit_stat,house_memb,chil_u_18_y,nation,religion,...,bk_acc,ins_prop,ins_life,ins_casco,health_ins,overdraft,cons_cred,mortgage,car_leas,pens_ins
0,10001,2,52,4,4,3,2,0,1,3,...,1,0,0,0,0,0,0,0,0,0
1,10002,2,66,3,3,4,4,0,1,3,...,1,0,0,0,0,0,0,0,0,0
2,10003,2,40,4,3,2,4,0,1,3,...,1,0,0,0,0,0,0,0,0,0
3,10005,2,54,3,4,2,4,0,1,4,...,1,0,0,0,0,0,0,0,1,0
4,10006,1,40,4,3,4,7,4,1,3,...,1,0,0,0,0,0,0,0,0,0


## Define Matching Keys

`left_on` and `right_on` specify the columns in each dataset to use as matching keys.

In [40]:
big5[0:5]

,age,gender,extrav,neurot,agreeab,conscient,openness,id
0,53,1,4.4,4.9,4.6,4.7,4.3,20001
1,46,2,2.2,2.9,3.5,4.2,2.6,20002
2,14,2,3.5,1.4,3.8,4.9,4.5,20003
3,19,2,2.2,1.7,3.7,2.6,4.1,20004
4,25,2,3.4,3.0,4.4,3.4,3.4,20005


## Generate Link Table

The link table shows all candidate matches with their fuzzy match scores.

In [41]:
left_on = ["sex", "age_sim"]
right_on = ["gender", "age"]

In [42]:
match_table = fm.link_table(demogr, big5, left_on, right_on, left_id_col='ind', right_id_col='id')

## Fuzzy Left Join

The left join appends the best-matching right-side record to each left-side record.

In [43]:
match_table[0:50]

,__id_left,__id_right,match_score,match_rank,sex,gender,age_sim,age
0,10001,27163,0.072145,1,2,2,52,52
1,10001,32407,0.071044,2,2,2,52,52
2,10001,31056,0.068877,3,2,2,52,52
3,10001,33594,0.068146,4,2,2,52,52
4,10001,28854,0.067393,5,2,2,52,52
5,10001,24299,0.067339,6,2,2,52,52
6,10001,31580,0.066097,7,2,2,52,52
7,10001,34235,0.065817,8,2,2,52,52
8,10001,34822,0.065565,9,2,2,52,52
9,10001,30840,0.065329,10,2,2,52,52


In [44]:
matched_results = fm.fuzzy_left_join(demogr, big5, left_on, right_on, left_id_col='ind', right_id_col='id')

## Profile Report

Generate a pandas-profiling report of the merged result.

In [45]:
matched_results[0:50]

,best_match_score,__id_left,__id_right,ind,sex,age_sim,lv_educ,empl_stat,marit_stat,house_memb,...,car_leas,pens_ins,age,gender,extrav,neurot,agreeab,conscient,openness,id
0,0.075021,10001,25135,10001,2,52,4,4,3,2,...,0,0,52,2,3.9,3.4,4.4,3.3,3.4,25135
48,0.078626,10002,27636,10002,2,66,3,3,4,4,...,0,0,66,2,4.3,4.4,4.9,4.1,4.3,27636
59,0.070214,10003,23546,10003,2,40,4,3,2,4,...,0,0,40,2,2.1,1.6,4.7,4.3,3.5,23546
109,0.070257,10005,29895,10005,2,54,3,4,2,4,...,1,0,54,2,2.5,3.1,3.5,3.0,2.8,29895
155,0.072985,10006,21525,10006,1,40,4,3,4,7,...,0,0,40,1,4.1,3.7,2.8,3.0,4.7,21525
205,0.055850,10007,20691,10007,2,27,4,1,1,1,...,1,0,27,2,2.8,3.4,3.9,3.6,4.2,20691
255,0.064302,10008,25603,10008,2,35,4,3,1,3,...,1,0,35,2,5.0,2.9,5.0,3.5,4.4,25603
305,0.072869,10010,30898,10010,1,59,4,3,2,3,...,0,0,59,1,3.9,4.6,4.4,3.8,4.2,30898
320,0.061419,10012,24242,10012,1,26,4,3,2,1,...,1,0,26,1,3.4,2.6,4.2,2.3,4.7,24242
370,0.070109,10014,23689,10014,2,60,4,4,2,4,...,0,0,60,2,4.1,3.6,4.8,3.3,5.0,23689


In [1]:
matched_results.profile_report()

NameError: name 'matched_results' is not defined

In [15]:
del matched_results['corp_oblig']

KeyError: "['corp_oblig'] not found in axis"

## Summary

The `fuzzymatcher` library reduces the O(n²) all-pairs comparison problem to a fast TF-IDF candidate lookup followed by exact scoring on a small candidate set. This makes it practical for datasets with thousands of records.

The match score in the link table can be used to filter low-confidence links before downstream analysis.